In [2]:
#!/usr/bin/env python3
import datetime
import json
import re
import shlex
import shutil
import subprocess
import textwrap
import time
from pathlib import Path
from typing import Optional, Tuple, Union


# =========================================================
# GRID5000 SETTINGS
# =========================================================
MODEL_ID = "openai/gpt-oss-120b"
SAFE_MODEL_NAME = re.sub(r"[^A-Za-z0-9]+", "_", MODEL_ID).strip("_")

WALLTIME = "48:00:00"
MAX_WAIT_MINUTES = 2880
POLL_SECONDS = 10
GPU_REQUEST = 2
EXPECTED_GPU_COUNT = 2
HOST_PREDICATE = "network_address like 'larochette-%'"

TORCH_INDEX_URL = "https://download.pytorch.org/whl/rocm6.1"
TORCH_VERSION = "2.5.1"
TORCHVISION_VERSION = "0.20.1"
TORCHAUDIO_VERSION = "2.5.1"
HSA_OVERRIDE_GFX_VERSION = "9.0.10"

SSH_HOST = "lux"



# Everything local goes under review_outputs
LOCAL_RESULTS_ROOT = Path("review_outputs")
LOCAL_JOBS_ROOT = LOCAL_RESULTS_ROOT / "jobs"
FINAL_OUTPUTS_ROOT = LOCAL_RESULTS_ROOT / "separated_reviews"

# Input PDFs are also inside review_outputs
PAPER_REVIEWS_ROOT = LOCAL_RESULTS_ROOT


class Grid5000Client:
    def __init__(self, ssh_host: str = SSH_HOST, local_results_root: Union[Path, str] = LOCAL_RESULTS_ROOT):
        self.ssh_host = ssh_host
        self.local_results_root = Path(local_results_root)
        self.local_results_root.mkdir(parents=True, exist_ok=True)

        self.remote_user: Optional[str] = None
        self.frontend_host: Optional[str] = None
        self.base_dir: Optional[str] = None

    def run(self, cmd: str, input_text: Optional[str] = None) -> str:
        cmd = textwrap.dedent(cmd).strip()
        if not cmd:
            return ""

        remote = "bash -lc " + shlex.quote(cmd)
        p = subprocess.run(
            ["ssh", self.ssh_host, remote],
            input=input_text,
            text=True,
            capture_output=True,
        )
        if p.returncode != 0:
            raise RuntimeError(
                "Remote cmd failed (rc="
                + str(p.returncode)
                + ").\n--- CMD ---\n"
                + cmd
                + "\n--- STDERR ---\n"
                + p.stderr
            )
        return p.stdout

    def run_ok(self, cmd: str) -> str:
        try:
            return self.run(cmd)
        except RuntimeError:
            return ""

    def ensure_identity(self) -> None:
        if self.remote_user is None:
            self.remote_user = self.run("id -un").strip()
        if self.frontend_host is None:
            self.frontend_host = self.run("hostname -f 2>/dev/null || hostname").strip()

    def cleanup_previous_artifacts(self) -> str:
        return self.run(
            r"""
            set -euo pipefail

            USERNAME="$(id -un)"
            HOME_DIR="$HOME"

            echo "[CLEANUP] removing old gpt-oss artifacts from HOME and /tmp"

            rm -rf "$HOME_DIR/.qa_launcher" 2>/dev/null || true
            mkdir -p "$HOME_DIR/.qa_launcher"

            rm -rf "$HOME_DIR/model-store/models" 2>/dev/null || true
            rm -rf "$HOME_DIR/model-store/hf-cache" 2>/dev/null || true
            rm -rf "$HOME_DIR/.cache/huggingface" 2>/dev/null || true
            rm -rf "$HOME_DIR/.cache/pip" 2>/dev/null || true
            rm -rf "$HOME_DIR/.cache/torch" 2>/dev/null || true
            rm -rf "$HOME_DIR/.cache/xdg" 2>/dev/null || true

            find "$HOME_DIR" -maxdepth 1 -type f -name 'qa_answer_*.txt' -delete 2>/dev/null || true

            rm -rf /tmp/"$USERNAME"-job-* 2>/dev/null || true
            rm -rf /tmp/"$USERNAME"-venv-* 2>/dev/null || true
            rm -rf /tmp/"$USERNAME"-qa-* 2>/dev/null || true
            rm -rf /tmp/"$USERNAME"-gptoss-* 2>/dev/null || true

            echo "[CLEANUP] done"
            echo "[CLEANUP] disk usage after cleanup:"
            du -sh "$HOME_DIR/.cache" 2>/dev/null || true
            df -h "$HOME_DIR" /tmp 2>/dev/null || true
            """
        )

    def resolve_base_dir(self) -> str:
        self.ensure_identity()
        print("[REMOTE] cleaning previous artifacts...", flush=True)
        print(self.cleanup_previous_artifacts().strip(), flush=True)

        self.base_dir = self.run("echo $HOME").strip()
        print("[REMOTE] using HOME:", self.base_dir, flush=True)
        return self.base_dir

    def make_workdir(self) -> str:
        if self.base_dir is None:
            self.resolve_base_dir()

        return self.run(
            r'''
            mkdir -p "$HOME/.qa_launcher"
            mktemp -d -p "$HOME/.qa_launcher" qa_run.XXXXXX
            '''
        ).strip()

    def upload_text(self, remote_dir: str, filename: str, content: str) -> str:
        remote_path = remote_dir + "/" + filename
        self.run(
            """
            set -euo pipefail
            cat > """
            + shlex.quote(remote_path)
            + """
            chmod 600 """
            + shlex.quote(remote_path),
            input_text=content,
        )
        return remote_path

    def mkdir_p(self, remote_path: str) -> None:
        self.run("mkdir -p " + shlex.quote(remote_path))

    def upload_file(self, local_path: Path, remote_path: str) -> str:
        local_path = Path(local_path)
        if not local_path.exists():
            raise FileNotFoundError("Missing local file: " + str(local_path.resolve()))

        remote_parent = str(Path(remote_path).parent)
        self.mkdir_p(remote_parent)

        p = subprocess.run(
            ["scp", str(local_path), self.ssh_host + ":" + remote_path],
            text=True,
            capture_output=True,
        )
        if p.returncode != 0:
            raise RuntimeError(
                "Failed to upload file.\n"
                + "local_path="
                + str(local_path.resolve())
                + "\nremote_path="
                + remote_path
                + "\n"
                + p.stderr
            )

        self.run("chmod 600 " + shlex.quote(remote_path))
        return remote_path

    def write_remote_script(self, remote_dir: str, script_name: str, script_text: str) -> str:
        remote_path = remote_dir + "/" + script_name
        self.run(
            "cat > "
            + shlex.quote(remote_path)
            + " <<'EOF'\n"
            + script_text
            + "\nEOF\nchmod 700 "
            + shlex.quote(remote_path)
        )
        return remote_path

    def submit_job(
        self,
        remote_dir: str,
        script_path: str,
        gpu_request: int,
        walltime: str,
        host_predicate: str,
    ) -> Tuple[str, str]:
        out = self.run(
            "oarsub -l "
            + shlex.quote("gpu=" + str(gpu_request) + ",walltime=" + walltime)
            + " -p "
            + shlex.quote(host_predicate)
            + " -d "
            + shlex.quote(remote_dir)
            + " -O "
            + shlex.quote(remote_dir + "/stdout.%jobid%.txt")
            + " -E "
            + shlex.quote(remote_dir + "/stderr.%jobid%.txt")
            + " "
            + shlex.quote(script_path)
        )
        m = re.search(r"OAR_JOB_ID=(\d+)", out)
        if not m:
            raise RuntimeError("Could not parse OAR_JOB_ID.\n--- oarsub output ---\n" + out)
        return m.group(1), out

    def oar_field(self, jobid: str, field: str) -> str:
        field = re.sub(r"[^A-Za-z0-9_]", "", field)
        return self.run_ok(
            "oarstat -j "
            + shlex.quote(str(jobid))
            + " -f 2>/dev/null | "
            + "sed -n 's/^[[:space:]]*"
            + field
            + "[[:space:]]*=[[:space:]]*//p' | "
            + "head -n 1 || true"
        ).strip()

    def job_state(self, jobid: str) -> str:
        return self.oar_field(jobid, "state") or "UNKNOWN"

    def job_host(self, jobid: str) -> str:
        return self.oar_field(jobid, "assigned_hostnames")

    def read_remote_text(self, path: str) -> str:
        return self.run_ok("cat " + shlex.quote(path) + " 2>/dev/null || true")

    def wait_for_answer(
        self,
        jobid: str,
        remote_dir: str,
        answer_path: str,
        status_path: str,
        poll_seconds: int,
        max_wait_minutes: int,
    ) -> Tuple[str, str]:
        deadline = time.time() + max_wait_minutes * 60
        final_state = "UNKNOWN"
        last_live_len = 0
        last_live_text = ""
        finishing_since = None

        def is_final_status(text: str) -> bool:
            status = (text or "").strip()
            return (
                status.startswith("OK")
                or status.startswith("PARTIAL_OK")
                or status.startswith("ERROR")
            )

        while time.time() < deadline:
            state = self.job_state(jobid)
            host = self.job_host(jobid)
            ts = datetime.datetime.now().strftime("%H:%M:%S")
            print("[" + ts + "] state=" + state + (" host=" + host if host else ""))
            final_state = state

            live_log_path = remote_dir + "/live.log"
            live_text = self.read_remote_text(live_log_path)
            if len(live_text) < last_live_len:
                last_live_len = 0
            if len(live_text) > last_live_len:
                new_live = live_text[last_live_len:]
                if new_live.strip():
                    print("\n[live.log]")
                    print(new_live, end="" if new_live.endswith("\n") else "\n", flush=True)
                last_live_len = len(live_text)
                last_live_text = live_text

            answer_text = self.read_remote_text(answer_path)
            status_text = self.read_remote_text(status_path)

            if is_final_status(status_text):
                if answer_text.strip():
                    return state, answer_text
                return state, status_text

            if state == "Finishing":
                if finishing_since is None:
                    finishing_since = time.time()
                if time.time() - finishing_since < 180:
                    time.sleep(poll_seconds)
                    continue

            if state in ("Terminated", "Error", "Unknown", "UNKNOWN"):
                if answer_text.strip():
                    return state, answer_text
                if status_text.strip():
                    return state, status_text
                if last_live_text.strip():
                    return state, last_live_text

            time.sleep(poll_seconds)

        answer_text = self.read_remote_text(answer_path)
        if answer_text.strip():
            return final_state, answer_text

        status_text = self.read_remote_text(status_path)
        if status_text.strip():
            return final_state, status_text

        return final_state, last_live_text

    def copy_remote_dir_to_local(self, remote_dir: str, local_dir: Path) -> Path:
        local_dir.mkdir(parents=True, exist_ok=True)
        rsync_cmd = [
            "rsync",
            "-az",
            "-e",
            "ssh",
            self.ssh_host + ":" + remote_dir + "/",
            str(local_dir) + "/",
        ]
        p = subprocess.run(rsync_cmd, text=True, capture_output=True)
        if p.returncode != 0:
            raise RuntimeError(
                "Failed to copy remote directory.\n"
                + "remote_dir="
                + remote_dir
                + "\nlocal_dir="
                + str(local_dir)
                + "\n"
                + p.stderr
            )
        return local_dir


def discover_review_files(root: Union[Path, str] = PAPER_REVIEWS_ROOT) -> list:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError("Missing reviews root folder: " + str(root.resolve()))

    files = []
    for p in sorted(root.rglob("submission_review*.pdf")):
        if not p.is_file():
            continue
        if ".ipynb_checkpoints" in p.parts:
            continue
        files.append(p.resolve())

    return files


def main():
    LOCAL_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL_JOBS_ROOT.mkdir(parents=True, exist_ok=True)
    FINAL_OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)

    review_files = discover_review_files()

    if not review_files:
        print("cwd:", Path.cwd())
        print("\nVisible top-level submission_review*.pdf:")
        for p in sorted(Path(".").glob("submission_review*.pdf")):
            print(" -", p.resolve())

        if PAPER_REVIEWS_ROOT.exists():
            print("\nVisible pdf files under " + str(PAPER_REVIEWS_ROOT.resolve()) + ":")
            for p in sorted(PAPER_REVIEWS_ROOT.rglob("*.pdf")):
                print(" -", p.resolve())

        raise FileNotFoundError(
            "No review files were found.\n"
            "Expected review PDFs like submission_review*.pdf under review_outputs/."
        )

    print("Found review files:")
    for p in review_files:
        print(" -", p)

    g5k = Grid5000Client()

    remote_base = g5k.resolve_base_dir()
    print("\nResolved remote base dir:", remote_base)

    remote_workdir = g5k.make_workdir()
    remote_inputs_dir = remote_workdir + "/input_reviews"
    remote_outputs_dir = remote_workdir + "/outputs"

    g5k.mkdir_p(remote_inputs_dir)
    g5k.mkdir_p(remote_outputs_dir)

    manifest = []
    for local_path in review_files:
        rel = local_path.relative_to(Path.cwd()).as_posix()
        remote_path = remote_inputs_dir + "/" + rel
        g5k.upload_file(local_path, remote_path)

        manifest.append({
            "source_file": rel,
            "remote_file": remote_path,
            "output_json_rel": rel + ".separated_reviews.json",
            "output_txt_rel": rel + ".separated_reviews.txt",
        })

    g5k.upload_text(remote_workdir, "manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))
    g5k.upload_text(remote_workdir, "model.txt", MODEL_ID)
    g5k.upload_text(remote_workdir, "torch_index_url.txt", TORCH_INDEX_URL)
    g5k.upload_text(remote_workdir, "torch_version.txt", TORCH_VERSION)
    g5k.upload_text(remote_workdir, "torchvision_version.txt", TORCHVISION_VERSION)
    g5k.upload_text(remote_workdir, "torchaudio_version.txt", TORCHAUDIO_VERSION)
    g5k.upload_text(remote_workdir, "hsa_override_gfx_version.txt", HSA_OVERRIDE_GFX_VERSION)
    g5k.upload_text(remote_workdir, "expected_gpu_count.txt", str(EXPECTED_GPU_COUNT) + "\n")

    run_sh = r"""#!/usr/bin/env bash
set -euo pipefail

WORKDIR="$PWD"
ANSWER_FILE="$WORKDIR/answer.txt"
STATUS_FILE="$WORKDIR/status.txt"
SUMMARY_FILE="$WORKDIR/summary.json"
LIVE_LOG="$WORKDIR/live.log"
: > "$LIVE_LOG"

exec > >(tee -a "$LIVE_LOG") 2>&1

echo "BOOTSTRAP_STARTED" > "$STATUS_FILE"

export MODEL_ID="$(cat "$WORKDIR/model.txt")"
export TORCH_INDEX_URL="$(cat "$WORKDIR/torch_index_url.txt")"
export TORCH_VERSION="$(cat "$WORKDIR/torch_version.txt")"
export TORCHVISION_VERSION="$(cat "$WORKDIR/torchvision_version.txt")"
export TORCHAUDIO_VERSION="$(cat "$WORKDIR/torchaudio_version.txt")"
export HSA_OVERRIDE_GFX_VERSION_VALUE="$(cat "$WORKDIR/hsa_override_gfx_version.txt")"
export EXPECTED_GPU_COUNT="$(cat "$WORKDIR/expected_gpu_count.txt")"

export JOB_TMP="/tmp/$USER-job-$OAR_JOB_ID"
export HF_HOME="$JOB_TMP/hf-home"
export HF_HUB_CACHE="$HF_HOME/hub"
export HF_DATASETS_CACHE="$JOB_TMP/datasets"
export XDG_CACHE_HOME="$JOB_TMP/xdg"
export PIP_CACHE_DIR="$JOB_TMP/pip"
export MODEL_LOAD_DIR="$JOB_TMP/models/model"
export OFFLOAD_DIR="$JOB_TMP/offload"

mkdir -p \
  "$HF_HUB_CACHE" \
  "$HF_DATASETS_CACHE" \
  "$XDG_CACHE_HOME" \
  "$PIP_CACHE_DIR" \
  "$MODEL_LOAD_DIR" \
  "$OFFLOAD_DIR"

unset HIP_VISIBLE_DEVICES || true
unset ROCR_VISIBLE_DEVICES || true
unset CUDA_VISIBLE_DEVICES || true
unset GPU_DEVICE_ORDINAL || true

export HSA_OVERRIDE_GFX_VERSION="$HSA_OVERRIDE_GFX_VERSION_VALUE"
export AMD_SERIALIZE_KERNEL=3
export TORCH_SHOW_CPP_STACKTRACES=1
export PYTORCH_HIP_ALLOC_CONF="expandable_segments:True,max_split_size_mb:128"
export TOKENIZERS_PARALLELISM=false
export PYTHONUNBUFFERED=1
export PYTHONFAULTHANDLER=1

echo "HOSTNAME=$(hostname)"
echo "WORKDIR=$WORKDIR"
echo "JOB_TMP=$JOB_TMP"
echo "HF_HOME=$HF_HOME"
echo "MODEL_LOAD_DIR=$MODEL_LOAD_DIR"
echo "OFFLOAD_DIR=$OFFLOAD_DIR"
echo "EXPECTED_GPU_COUNT=$EXPECTED_GPU_COUNT"
echo "CUDA_VISIBLE_DEVICES=${CUDA_VISIBLE_DEVICES-}"
echo "HIP_VISIBLE_DEVICES=${HIP_VISIBLE_DEVICES-}"
echo "ROCR_VISIBLE_DEVICES=${ROCR_VISIBLE_DEVICES-}"
echo "[OAR ENV]"
env | sort | grep '^OAR_' || true
echo "[DEVICE NODES]"
ls -l /dev/kfd /dev/dri/render* 2>/dev/null || true

VENV="/tmp/$USER-venv-$OAR_JOB_ID"

cleanup_tmp() {
  RC=$?
  echo "[CLEANUP] removing current job tmp files"
  rm -rf "$VENV" 2>/dev/null || true
  rm -rf "$JOB_TMP" 2>/dev/null || true
  exit $RC
}
trap cleanup_tmp EXIT

python3 -m venv "$VENV"
source "$VENV/bin/activate"

python -m pip -q install -U pip wheel setuptools
python -m pip -q install --no-cache-dir \
  "torch==${TORCH_VERSION}" \
  "torchvision==${TORCHVISION_VERSION}" \
  "torchaudio==${TORCHAUDIO_VERSION}" \
  --index-url "${TORCH_INDEX_URL}"

python -m pip -q install --no-cache-dir \
  "transformers>=4.52" \
  "accelerate>=1.7" \
  "safetensors>=0.4.3" \
  "sentencepiece" \
  "huggingface_hub>=0.23" \
  "pypdf"

python - <<'PY'
import os
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=os.environ["MODEL_ID"],
    local_dir=os.environ["MODEL_LOAD_DIR"],
)
print("MODEL downloaded to:", os.environ["MODEL_LOAD_DIR"])
PY

du -sh "$MODEL_LOAD_DIR" 2>/dev/null || true

python - <<'PY'
from pathlib import Path
import ast
import gc
import json
import os
import re
import traceback

import torch
from pypdf import PdfReader
from transformers import AutoTokenizer, AutoModelForCausalLM

workdir = Path.cwd()
manifest = json.loads((workdir / "manifest.json").read_text(encoding="utf-8"))
outputs_dir = workdir / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

answer_file = workdir / "answer.txt"
status_file = workdir / "status.txt"
summary_file = workdir / "summary.json"

model_id = os.environ["MODEL_ID"]
model_path = os.environ["MODEL_LOAD_DIR"]
offload_dir = Path(os.environ["OFFLOAD_DIR"])
expected_gpu_count = int(os.environ["EXPECTED_GPU_COUNT"])

STRUCTURED_SECTION_ALIASES = {
    "general comments and summary of recommendation": "General comments and summary of recommendation",
    "general comments": "General comments and summary of recommendation",
    "summary of recommendation": "General comments and summary of recommendation",
    "content": "Content",
    "structure and argument": "Structure and argument",
    "structure & argument": "Structure and argument",
    "figures/tables": "Figures/tables",
    "figures / tables": "Figures/tables",
    "figures and tables": "Figures/tables",
    "language": "Language",
    "ethical approval": "Ethical approval",
    "data / primary sources availability": "Data / primary sources availability",
    "data/primary sources availability": "Data / primary sources availability",
    "data and primary sources availability": "Data / primary sources availability",
    "data availability": "Data / primary sources availability",
    "primary sources availability": "Data / primary sources availability",
    "comments to the editor": "Comments to the editor",
}

STRUCTURED_SECTION_ORDER = [
    "General comments and summary of recommendation",
    "Content",
    "Structure and argument",
    "Figures/tables",
    "Language",
    "Ethical approval",
    "Data / primary sources availability",
    "Comments to the editor",
]
STRUCTURED_ORDER_INDEX = {name: i for i, name in enumerate(STRUCTURED_SECTION_ORDER)}

METADATA_KEYS = {
    "reviewer_id",
    "reviewer",
    "score",
    "overall_score",
    "overall_rating",
    "recommendation",
    "final_score",
    "score_raw",
    "round",
    "editor",
    "paper_id",
    "submission_id",
    "review_files",
}

PROMPT_STARTERS = (
    "describe ",
    "please ",
    "comment ",
    "evaluate ",
    "discuss ",
    "is there ",
    "are the authors ",
    "does the ",
    "does this ",
    "has the ",
    "should the ",
    "should this ",
    "please consider ",
    "please comment ",
    "please discuss ",
    "please describe ",
    "please examine ",
    "summarise ",
    "summarize ",
    "consider whether ",
    "consider the ",
    "comment on ",
    "examine the ",
    "is the ",
    "are there ",
    "to what extent ",
    "overall, is ",
    "overall is ",
)

PROMPT_KEY_PHRASES = (
    "please consider",
    "please comment",
    "please discuss",
    "please describe",
    "please examine",
    "comment on the",
    "comment on whether",
    "evaluate the",
    "discuss the",
    "is there enough",
    "are the authors",
    "does the manuscript",
    "does the article",
    "does the paper",
    "has the author",
    "has the manuscript",
    "is the submission",
    "is the manuscript",
    "is the article",
    "is the paper",
    "is all statistical analysis",
    "overall, is the submission",
    "overall is the submission",
)

COMMENT_STARTERS = (
    "i ",
    "i'm ",
    "in my ",
    "my ",
    "the paper ",
    "this paper ",
    "the article ",
    "this article ",
    "the manuscript ",
    "this manuscript ",
    "the author ",
    "the authors ",
    "overall, the ",
    "overall the ",
    "one concern",
    "one issue",
    "my main ",
    "i recommend",
    "i find",
    "i found",
    "i have",
    "i would",
    "it would",
    "there is ",
    "there are ",
)

COMMENT_SIGNAL_PHRASES = (
    "the article",
    "the paper",
    "the manuscript",
    "the author",
    "the authors",
    "this article",
    "this paper",
    "this manuscript",
    "in my view",
    "in my opinion",
    "i think",
    "i find",
    "i found",
    "i would",
    "i have",
    "my concern",
    "my main concern",
    "one concern",
    "one issue",
    "should be revised",
    "is convincing",
    "is not convincing",
    "is well written",
    "is unclear",
    "is clear",
    "is original",
)

SECTION_SPLIT_SYSTEM_PROMPT = (
    "Reasoning: low\n"
    "You are splitting one peer-review section into exactly TWO parts.\n\n"
    "Output keys:\n"
    "1. question\n"
    "2. reviewer_comment\n\n"
    "Rules:\n"
    "- Detect the leading copied question or prompt at the START of the section.\n"
    "- Everything after that leading question or prompt must be reviewer_comment.\n"
    "- reviewer_comment is simply the REST OF THE TEXT after the question.\n"
    "- Do not paraphrase.\n"
    "- Do not rewrite.\n"
    "- Copy text from source only.\n"
    "- If there is no leading copied prompt, use section_heading as question and full section_text as reviewer_comment.\n"
    "- Return strict JSON only.\n\n"
    "Output schema:\n"
    "{\"question\":\"string\",\"reviewer_comment\":\"string\"}"
)

FULL_REVIEW_SYSTEM_PROMPT = (
    "Reasoning: low\n"
    "You are extracting separate review items from one peer-review file.\n\n"
    "Goal:\n"
    "- Split the review into distinct review items.\n"
    "- For each review item, return:\n"
    "  1. the review question or review prompt that the reviewer is answering\n"
    "  2. the reviewer comment text\n\n"
    "Rules:\n"
    "- Return STRICT JSON only.\n"
    "- Do not include markdown fences.\n"
    "- Do not summarize if the original wording is available.\n"
    "- Preserve original text as much as possible, only clean obvious whitespace.\n"
    "- If a review item has no explicit question, set question to an empty string.\n"
    "- If a section body starts with copied editorial instructions, rubric text, or a quoted review prompt, move that text into question.\n"
    "- Do not keep copied prompt text inside reviewer_comment.\n"
    "- In structured editorial forms, prefer the embedded prompt as question; if none exists, use the section heading.\n"
    "- Keep one item per real review point.\n"
    "- Ignore numeric scores unless they are part of reviewer comment text.\n\n"
    "Output schema:\n"
    "{\n"
    "  \"items\": [\n"
    "    {\n"
    "      \"question\": \"string\",\n"
    "      \"reviewer_comment\": \"string\"\n"
    "    }\n"
    "  ]\n"
    "}"
)


def atomic_write(path: Path, content: str) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(content, encoding="utf-8")
    tmp.replace(path)


def atomic_json(path: Path, obj) -> None:
    atomic_write(path, json.dumps(obj, indent=2, ensure_ascii=False) + "\n")


def clean_text(text) -> str:
    if text is None:
        return ""
    text = str(text).replace("\r\n", "\n").replace("\r", "\n").replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def preview_text(text: str, limit: int = 5000) -> str:
    txt = str(text or "").replace("\r\n", "\n").replace("\r", "\n").strip()
    if len(txt) <= limit:
        return txt
    return txt[:limit] + "\n...[truncated]..."


def normalize_heading(text: str) -> str:
    text = clean_text(text)
    text = re.sub(r"^\s*\d+\s*[\)\.\-:]*\s*", "", text)
    text = text.strip().strip(":").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def canonical_heading(text: str) -> str:
    text = normalize_heading(text).lower()
    text = text.replace("&", "and")
    text = re.sub(r"\s*/\s*", " / ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_for_compare(text: str) -> str:
    text = clean_text(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def preserved_in_source(candidate: str, source: str) -> bool:
    cand = normalize_for_compare(candidate)
    src = normalize_for_compare(source)
    if not cand:
        return False
    return cand in src


def parse_json_like(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return ast.literal_eval(text)


def strip_gptoss_wrappers(text: str) -> str:
    raw = clean_text(text)
    if not raw:
        return raw

    final_marker = "<|channel|>final<|message|>"
    if final_marker in raw:
        raw = raw.split(final_marker, 1)[1]

    if "<|message|>" in raw and final_marker not in text:
        raw = raw.split("<|message|>", 1)[-1]

    for stop_marker in ("<|end|>", "<|return|>", "<|eot_id|>"):
        if stop_marker in raw:
            raw = raw.split(stop_marker, 1)[0]

    return raw.strip()


def match_structured_heading(line: str):
    line = clean_text(line)
    if not line:
        return None

    numbered_match = re.match(r"^\s*(\d+)\s*[\)\.\-:]*\s*(.*)$", line)
    line_wo_num = numbered_match.group(2).strip() if numbered_match else line
    line_wo_num_canon = canonical_heading(line_wo_num)

    for alias, display in STRUCTURED_SECTION_ALIASES.items():
        if re.fullmatch(rf"{re.escape(alias)}", line_wo_num_canon, flags=re.IGNORECASE):
            return display, ""

        if re.match(rf"^{re.escape(alias)}\s*:", line_wo_num_canon, flags=re.IGNORECASE):
            alias_pattern = rf"^\s*(?:\d+\s*[\)\.\-:]*\s*)?{re.escape(alias)}\s*:?\s*(.*)$"
            alias_m = re.match(alias_pattern, line, flags=re.IGNORECASE)
            rest = clean_text(alias_m.group(1)) if alias_m else ""
            return display, rest

        display_pattern = rf"^\s*(?:\d+\s*[\)\.\-:]*\s*)?{re.escape(display)}\s*:?\s*(.*)$"
        display_m = re.match(display_pattern, line, flags=re.IGNORECASE)
        if display_m:
            rest = clean_text(display_m.group(1))
            return display, rest

    return None


def extract_structured_items(review_text: str):
    lines = review_text.replace("\r\n", "\n").replace("\r", "\n").split("\n")

    items = []
    current_title = None
    current_lines = []

    def flush_current():
        nonlocal current_title, current_lines, items
        if not current_title:
            return
        comment = clean_text("\n".join(current_lines))
        if comment:
            items.append({
                "question": current_title,
                "reviewer_comment": comment,
            })
        current_title = None
        current_lines = []

    for raw_line in lines:
        line = clean_text(raw_line)
        hit = match_structured_heading(line)

        if hit:
            flush_current()
            current_title, first_rest = hit
            current_lines = [first_rest] if first_rest else []
        else:
            if current_title:
                current_lines.append(raw_line)

    flush_current()

    if len(items) >= 2:
        items.sort(key=lambda x: STRUCTURED_ORDER_INDEX.get(x["question"], 999))
        return items

    return []


def extract_json_block(text: str) -> str:
    if text is None:
        raise ValueError("Empty model output.")

    raw = strip_gptoss_wrappers(str(text))
    if not raw:
        raise ValueError("Empty model output.")

    candidates = []

    fence_match = re.search(r"```(?:json)?\s*(.*?)\s*```", raw, flags=re.DOTALL | re.IGNORECASE)
    if fence_match:
        candidates.append(fence_match.group(1).strip())

    cleaned = re.sub(r"^\s*json\s*[:\-]?\s*", "", raw, flags=re.IGNORECASE)
    if cleaned != raw:
        candidates.append(cleaned.strip())

    candidates.append(raw)

    start_positions = [i for i, ch in enumerate(raw) if ch in "[{"]
    for start in start_positions:
        opener = raw[start]
        closer = "}" if opener == "{" else "]"
        depth = 0
        in_string = False
        escape = False

        for i in range(start, len(raw)):
            ch = raw[i]

            if in_string:
                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == '"':
                    in_string = False
                continue

            if ch == '"':
                in_string = True
            elif ch == opener:
                depth += 1
            elif ch == closer:
                depth -= 1
                if depth == 0:
                    candidates.append(raw[start:i + 1])
                    break

    seen = set()
    for candidate in candidates:
        candidate = candidate.strip()
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)

        try:
            parse_json_like(candidate)
            return candidate
        except Exception:
            pass

    raise ValueError(
        "No JSON block found in model output.\n\n"
        + "RAW MODEL OUTPUT:\n"
        + raw[:4000]
    )


def first_nonempty(*values) -> str:
    for value in values:
        txt = clean_text(value)
        if txt:
            return txt
    return ""


def item_from_single_key_dict(item):
    if not isinstance(item, dict):
        return None

    meaningful_keys = []
    for k, v in item.items():
        key = clean_text(k)
        if not key:
            continue
        if canonical_heading(key) in METADATA_KEYS:
            continue
        if isinstance(v, (str, int, float)) and clean_text(v):
            meaningful_keys.append((key, clean_text(v)))

    if len(meaningful_keys) == 1:
        q, c = meaningful_keys[0]
        return {"question": q, "reviewer_comment": c}

    return None


def coerce_items_from_dict(data: dict):
    for key in ("items", "review_items", "comments", "sections", "review_sections"):
        value = data.get(key)
        if isinstance(value, list):
            return value

    direct_structured = []
    for key, value in data.items():
        canon = canonical_heading(key)
        if canon in STRUCTURED_SECTION_ALIASES and clean_text(value):
            display = STRUCTURED_SECTION_ALIASES[canon]
            direct_structured.append({
                "question": display,
                "reviewer_comment": clean_text(value),
            })

    if direct_structured:
        direct_structured.sort(key=lambda x: STRUCTURED_ORDER_INDEX.get(x["question"], 999))
        return direct_structured

    loose_items = []
    for key, value in data.items():
        canon = canonical_heading(key)
        if canon in METADATA_KEYS:
            continue
        if isinstance(value, (str, int, float)) and clean_text(value):
            loose_items.append({
                "question": clean_text(key),
                "reviewer_comment": clean_text(value),
            })

    if loose_items:
        return loose_items

    return []


def normalize_payload(raw: str):
    block = extract_json_block(raw)
    data = parse_json_like(block)

    if isinstance(data, list):
        items = data
    elif isinstance(data, dict):
        items = coerce_items_from_dict(data)
    else:
        raise ValueError("Model output is neither a list nor an object.")

    if not isinstance(items, list):
        raise ValueError("Normalized items is not a list.")

    out_items = []
    for idx, item in enumerate(items, start=1):
        if isinstance(item, dict):
            question = first_nonempty(
                item.get("question"),
                item.get("prompt"),
                item.get("title"),
                item.get("heading"),
                item.get("section"),
                item.get("criterion"),
            )
            reviewer_comment = first_nonempty(
                item.get("reviewer_comment"),
                item.get("comment"),
                item.get("response"),
                item.get("answer"),
                item.get("text"),
                item.get("content"),
                item.get("value"),
            )

            if not question and not reviewer_comment:
                single = item_from_single_key_dict(item)
                if single:
                    question = single["question"]
                    reviewer_comment = single["reviewer_comment"]
        else:
            question = ""
            reviewer_comment = clean_text(item)

        if not question and not reviewer_comment:
            continue

        out_items.append({
            "item_id": idx,
            "question": question,
            "reviewer_comment": reviewer_comment,
        })

    return {"items": out_items}


def strip_outer_quotes(text: str) -> str:
    t = clean_text(text)
    if len(t) >= 2 and ((t[0] == '"' and t[-1] == '"') or (t[0] == "'" and t[-1] == "'")):
        return t[1:-1].strip()
    return t


def split_sentences(text: str):
    raw = clean_text(text)
    if not raw:
        return []
    parts = re.split(r"(?<=[\.\?\!])\s+", raw)
    return [clean_text(p) for p in parts if clean_text(p)]


def looks_like_prompt_sentence(text: str) -> bool:
    t = strip_outer_quotes(text).lower()
    if not t:
        return False

    if t.startswith(PROMPT_STARTERS):
        return True

    if any(phrase in t for phrase in PROMPT_KEY_PHRASES):
        return True

    if t.endswith("?"):
        if any(word in t for word in (
            "submission",
            "paper",
            "article",
            "manuscript",
            "author",
            "authors",
            "abstract",
            "introduction",
            "conclusion",
            "method",
            "methods",
            "analysis",
            "data",
            "evidence",
            "argument",
            "referencing",
            "structure",
            "language",
        )):
            return True

    return False


def looks_like_reviewer_comment_sentence(text: str) -> bool:
    t = strip_outer_quotes(text).lower()
    if not t:
        return False

    if t.startswith(COMMENT_STARTERS):
        return True

    if any(phrase in t for phrase in COMMENT_SIGNAL_PHRASES):
        return True

    if re.match(r"^(good|strong|weak|poor|excellent|interesting|convincing|unconvincing|clear|unclear)\b", t):
        return True

    return False


def looks_like_prompt(text: str) -> bool:
    t = clean_text(text)
    if not t:
        return False

    paragraphs = [clean_text(p) for p in re.split(r"\n\s*\n", t) if clean_text(p)]
    first_para = paragraphs[0] if paragraphs else t

    first_sentences = split_sentences(first_para)[:3]
    if not first_sentences:
        return False

    prompt_hits = sum(1 for s in first_sentences if looks_like_prompt_sentence(s))
    comment_hits = sum(1 for s in first_sentences if looks_like_reviewer_comment_sentence(s))

    if prompt_hits >= 1 and prompt_hits >= comment_hits:
        return True

    lower_first = strip_outer_quotes(first_para).lower()
    if lower_first.startswith(PROMPT_STARTERS):
        return True

    return False


def looks_like_reviewer_comment(text: str) -> bool:
    t = clean_text(text)
    if not t:
        return False

    first_sentences = split_sentences(t)[:3]
    if not first_sentences:
        return False

    comment_hits = sum(1 for s in first_sentences if looks_like_reviewer_comment_sentence(s))
    prompt_hits = sum(1 for s in first_sentences if looks_like_prompt_sentence(s))

    return comment_hits >= 1 and comment_hits >= prompt_hits


def find_question_boundary(section_heading: str, section_text: str) -> dict:
    raw = str(section_text or "").replace("\r\n", "\n").replace("\r", "\n").strip()
    raw_clean = clean_text(raw)

    default = {
        "question": clean_text(section_heading),
        "reviewer_comment": raw_clean,
    }

    if not raw_clean:
        return default

    paragraphs = [clean_text(p) for p in re.split(r"\n\s*\n", raw) if clean_text(p)]

    # Case 1: first paragraph is the copied prompt, everything else is the comment
    if len(paragraphs) >= 2:
        first_para = paragraphs[0]
        rest = "\n\n".join(paragraphs[1:]).strip()
        if (
            first_para
            and rest
            and looks_like_prompt(first_para)
            and not looks_like_reviewer_comment(first_para)
            and preserved_in_source(rest, raw)
        ):
            return {
                "question": first_para,
                "reviewer_comment": rest,
            }

    # Case 2: prompt and comment are in same paragraph; take leading prompt sentences,
    # and the rest is the comment.
    sentences = split_sentences(raw_clean)
    if len(sentences) >= 2:
        prompt_sentences = []
        split_idx = None

        for idx, sentence in enumerate(sentences):
            if looks_like_prompt_sentence(sentence) and not looks_like_reviewer_comment_sentence(sentence):
                prompt_sentences.append(sentence)
                continue

            if prompt_sentences:
                split_idx = idx
                break

            # first sentence is not prompt -> no deterministic split
            break

        if prompt_sentences and split_idx is not None:
            question = clean_text(" ".join(sentences[:split_idx]))
            reviewer_comment = clean_text(" ".join(sentences[split_idx:]))

            if (
                question
                and reviewer_comment
                and looks_like_prompt(question)
                and preserved_in_source(reviewer_comment, raw)
            ):
                return {
                    "question": question,
                    "reviewer_comment": reviewer_comment,
                }

    # Case 3: if a prompt block contains question marks, split after the last
    # prompt-looking question mark near the start.
    search_text = raw[:2500]
    qmark_positions = [m.end() for m in re.finditer(r"\?", search_text)]
    for end_pos in reversed(qmark_positions):
        question_part = clean_text(raw[:end_pos])
        comment_part = clean_text(raw[end_pos:])
        if (
            question_part
            and comment_part
            and looks_like_prompt(question_part)
            and preserved_in_source(comment_part, raw)
        ):
            return {
                "question": question_part,
                "reviewer_comment": comment_part,
            }

    return default


def fallback_single_item(review_text: str) -> dict:
    text = clean_text(review_text)
    if not text:
        return {"items": []}

    return {
        "items": [
            {
                "item_id": 1,
                "question": "",
                "reviewer_comment": text,
            }
        ]
    }


def pick_input_device(model):
    device_map = getattr(model, "hf_device_map", None)
    if isinstance(device_map, dict):
        for _, v in device_map.items():
            if isinstance(v, int):
                return torch.device(f"cuda:{v}")
            if isinstance(v, str) and v not in ("cpu", "disk"):
                return torch.device(v)
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    return torch.device("cpu")


diag = {
    "model_id": model_id,
    "model_path": model_path,
    "offload_dir": str(offload_dir),
    "torch": {},
}


def write_progress(results, errors, current_file="", current_stage="RUNNING") -> None:
    summary = {
        "processed_count": len(results),
        "error_count": len(errors),
        "total_files": len(manifest),
        "current_file": current_file,
        "current_stage": current_stage,
        "results": results,
        "errors": errors,
        "diagnostics": diag,
    }

    atomic_json(summary_file, summary)
    atomic_json(answer_file, summary)

    atomic_write(
        status_file,
        current_stage
        + "\nprocessed="
        + str(len(results))
        + "/"
        + str(len(manifest))
        + "\nerrors="
        + str(len(errors))
        + "\ncurrent_file="
        + current_file
        + "\n",
    )

    try:
        home_answer = Path.home() / ("qa_answer_" + os.environ["OAR_JOB_ID"] + ".txt")
        home_answer.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    except Exception:
        pass


def model_generate_json(
    tokenizer,
    model,
    system_prompt: str,
    user_prompt: str,
    max_input_tokens: int,
    max_new_tokens: int,
    debug_label: str = "",
):
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = system_prompt + "\n\n" + user_prompt

    if debug_label:
        print("\n" + "=" * 100, flush=True)
        print("PROMPT :: " + debug_label, flush=True)
        print("-" * 100, flush=True)
        print(preview_text(user_prompt, 5000), flush=True)
        print("=" * 100, flush=True)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens,
    )

    input_device = pick_input_device(model)
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else None,
        use_cache=True,
    )

    input_len = inputs["input_ids"].shape[1]
    gen_ids = output[0][input_len:]
    raw_answer = tokenizer.decode(gen_ids, skip_special_tokens=False).strip()

    if debug_label:
        print("\n" + "=" * 100, flush=True)
        print("RAW MODEL ANSWER :: " + debug_label, flush=True)
        print("-" * 100, flush=True)
        print(preview_text(raw_answer, 5000), flush=True)
        print("=" * 100, flush=True)

    block = extract_json_block(raw_answer)
    parsed = parse_json_like(block)

    if debug_label:
        print("PARSED JSON OK :: " + debug_label, flush=True)

    return parsed


def split_structured_item_with_model(
    tokenizer,
    model,
    section_heading: str,
    section_text: str,
    debug_label: str = "",
) -> dict:
    # Deterministic only:
    # find the question correctly, then the rest of text is the comment.
    split_item = find_question_boundary(section_heading, section_text)

    if debug_label:
        print("[RULE-BASED SPLIT USED]", flush=True)

    return split_item


def split_full_review_with_model(tokenizer, model, source_file: str, review_text: str, debug_label: str = "") -> dict:
    user_prompt = (
        "Split this review file into distinct review items.\n\n"
        "Important:\n"
        "There are at least two common review structures.\n\n"
        "Structure A:\n"
        "- The review is already organized as separate question/answer style review points.\n\n"
        "Structure B:\n"
        "- The review uses fixed editorial sections such as:\n"
        "  - General comments and summary of recommendation\n"
        "  - Content\n"
        "  - Structure and argument\n"
        "  - Figures/tables\n"
        "  - Language\n"
        "  - Ethical approval\n"
        "  - Data / primary sources availability\n"
        "  - Comments to the editor\n"
        "- In that case, EACH section must become one item.\n"
        "- If a section body starts with copied editorial instructions, rubric text, or a quoted review prompt, move that text into question.\n"
        "- Do not keep copied prompt text inside reviewer_comment.\n"
        "- If there is no embedded prompt, use the section heading as question.\n"
        "- Do not drop short sections just because they are brief.\n\n"
        "Review file path:\n"
        + source_file
        + "\n\n"
        + "Review file content:\n"
        + "\"\"\"\n"
        + review_text
        + "\n\"\"\"\n"
    )

    try:
        data = model_generate_json(
            tokenizer=tokenizer,
            model=model,
            system_prompt=FULL_REVIEW_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            max_input_tokens=12000,
            max_new_tokens=700,
            debug_label=debug_label,
        )
    except Exception:
        return fallback_single_item(review_text)

    if isinstance(data, dict) and "items" in data and isinstance(data["items"], list):
        items = data["items"]
    else:
        try:
            normalized = normalize_payload(json.dumps(data, ensure_ascii=False))
            items = normalized["items"]
        except Exception:
            return fallback_single_item(review_text)

    out_items = []
    for idx, item in enumerate(items, start=1):
        if not isinstance(item, dict):
            continue

        question = first_nonempty(
            item.get("question"),
            item.get("prompt"),
            item.get("title"),
            item.get("heading"),
            item.get("section"),
            item.get("criterion"),
        )
        reviewer_comment = first_nonempty(
            item.get("reviewer_comment"),
            item.get("comment"),
            item.get("response"),
            item.get("answer"),
            item.get("text"),
            item.get("content"),
            item.get("value"),
        )

        if not question and not reviewer_comment:
            continue

        out_items.append({
            "item_id": idx,
            "question": question,
            "reviewer_comment": reviewer_comment,
        })

    if not out_items:
        return fallback_single_item(review_text)

    return {"items": out_items}


def render_txt(source_file: str, payload: dict) -> str:
    lines = ["Source file: " + source_file, ""]

    for item in payload["items"]:
        lines.append("=== REVIEW ITEM " + str(item["item_id"]) + " ===")
        lines.append("Question:")
        lines.append(item["question"] if item["question"] else "[NO EXPLICIT QUESTION FOUND]")
        lines.append("")
        lines.append("Reviewer comment:")
        lines.append(item["reviewer_comment"])
        lines.append("")

    if len(lines) == 2:
        lines.append("[NO REVIEW ITEMS FOUND]")

    return "\n".join(lines).rstrip() + "\n"


try:
    if not torch.cuda.is_available():
        raise RuntimeError("torch.cuda.is_available() is False")

    if torch.cuda.device_count() != expected_gpu_count:
        raise RuntimeError(
            f"Expected {expected_gpu_count} visible GPUs inside the job, "
            f"but found {torch.cuda.device_count()}."
        )

    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
        use_fast=False,
        padding_side="left",
    )

    if getattr(tokenizer, "pad_token_id", None) is None:
        tokenizer.pad_token = tokenizer.eos_token

    max_memory = {0: "50GiB", 1: "50GiB", "cpu": "300GiB"}

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        device_map="auto",
        max_memory=max_memory,
        use_safetensors=True,
        torch_dtype="auto",
        low_cpu_mem_usage=True,
        attn_implementation="eager",
        offload_folder=str(offload_dir),
        offload_state_dict=True,
    )
    model.eval()

    diag["torch"] = {
        "torch_version": torch.__version__,
        "torch_cuda_version": torch.version.cuda,
        "torch_hip_version": torch.version.hip,
        "cuda_is_available": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count(),
        "device_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        "max_memory": max_memory,
        "hf_device_map": getattr(model, "hf_device_map", None),
    }

    results = []
    errors = []

    write_progress(results, errors, current_file="__bootstrap__", current_stage="MODEL_LOADED")

    for file_idx, entry in enumerate(manifest, start=1):
        source_file = entry["source_file"]
        pdf_path = Path(entry["remote_file"])

        print("\n" + "=" * 100, flush=True)
        print("START FILE " + str(file_idx) + "/" + str(len(manifest)) + " :: " + source_file, flush=True)
        print("=" * 100, flush=True)

        write_progress(
            results,
            errors,
            current_file=source_file,
            current_stage="RUNNING_FILE_" + str(file_idx) + "_OF_" + str(len(manifest)),
        )

        try:
            reader = PdfReader(str(pdf_path))
            review_text = "\n".join(page.extract_text() or "" for page in reader.pages).strip()

            print("PDF pages: " + str(len(reader.pages)), flush=True)
            print("Extracted chars: " + str(len(review_text)), flush=True)

            if not review_text:
                raise ValueError("No extractable text found in PDF: " + str(pdf_path))

            structured_items = extract_structured_items(review_text)

            if structured_items:
                print("Structured sections detected: " + str(len(structured_items)), flush=True)

                cleaned_items = []
                for idx, item in enumerate(structured_items, start=1):
                    print("\n" + "-" * 100, flush=True)
                    print("SECTION " + str(idx) + "/" + str(len(structured_items)) + " :: " + item["question"], flush=True)
                    print("[SECTION INPUT]", flush=True)
                    print(preview_text(item["reviewer_comment"], 2500), flush=True)

                    split_item = split_structured_item_with_model(
                        tokenizer=tokenizer,
                        model=model,
                        section_heading=item["question"],
                        section_text=item["reviewer_comment"],
                        debug_label=source_file + " :: structured_section_" + str(idx) + " :: " + item["question"],
                    )

                    print("[FINAL QUESTION]", flush=True)
                    print(preview_text(split_item["question"], 1500), flush=True)
                    print("[FINAL COMMENT]", flush=True)
                    print(preview_text(split_item["reviewer_comment"], 2500), flush=True)

                    cleaned_items.append({
                        "item_id": idx,
                        "question": split_item["question"],
                        "reviewer_comment": split_item["reviewer_comment"],
                    })

                payload = {"items": cleaned_items}
                print(
                    "PROCESSED "
                    + source_file
                    + " -> "
                    + str(len(payload["items"]))
                    + " items [rule_based_structured]",
                    flush=True,
                )
            else:
                payload = split_full_review_with_model(
                    tokenizer=tokenizer,
                    model=model,
                    source_file=source_file,
                    review_text=review_text,
                    debug_label=source_file + " :: full_review",
                )

                if not payload["items"]:
                    raise ValueError("No review items extracted from model output for: " + source_file)

                print(
                    "PROCESSED "
                    + source_file
                    + " -> "
                    + str(len(payload["items"]))
                    + " items [model_based]",
                    flush=True,
                )

                for item in payload["items"]:
                    print("\n" + "-" * 100, flush=True)
                    print("ITEM " + str(item["item_id"]), flush=True)
                    print("[QUESTION]", flush=True)
                    print(preview_text(item["question"], 1500) if item["question"] else "[NO EXPLICIT QUESTION FOUND]", flush=True)
                    print("[REVIEWER COMMENT]", flush=True)
                    print(preview_text(item["reviewer_comment"], 2500), flush=True)

            json_out = outputs_dir / entry["output_json_rel"]
            txt_out = outputs_dir / entry["output_txt_rel"]
            json_out.parent.mkdir(parents=True, exist_ok=True)
            txt_out.parent.mkdir(parents=True, exist_ok=True)

            atomic_json(json_out, payload)
            atomic_write(txt_out, render_txt(source_file, payload))

            results.append({
                "source_file": source_file,
                "item_count": len(payload["items"]),
                "json_output": str(json_out.relative_to(workdir)),
                "txt_output": str(txt_out.relative_to(workdir)),
            })

            write_progress(
                results,
                errors,
                current_file=source_file,
                current_stage="DONE_FILE_" + str(file_idx) + "_OF_" + str(len(manifest)),
            )

        except Exception as e:
            err = {
                "source_file": source_file,
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
            errors.append(err)

            err_out = outputs_dir / (source_file + ".separated_reviews.error.txt")
            err_out.parent.mkdir(parents=True, exist_ok=True)
            atomic_write(err_out, err["traceback"])

            print("FAILED " + source_file, flush=True)
            print(err["traceback"], flush=True)

            write_progress(
                results,
                errors,
                current_file=source_file,
                current_stage="FAILED_FILE_" + str(file_idx) + "_OF_" + str(len(manifest)),
            )

        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    final_stage = "PARTIAL_OK" if errors else "OK"
    write_progress(results, errors, current_file="", current_stage=final_stage)

except Exception:
    err = {
        "processed_count": 0,
        "error_count": 1,
        "results": [],
        "errors": [
            {
                "source_file": "__bootstrap__",
                "error": "fatal bootstrap/model load error",
                "traceback": traceback.format_exc(),
            }
        ],
        "diagnostics": diag,
    }
    atomic_json(summary_file, err)
    atomic_json(answer_file, err)
    atomic_write(status_file, "ERROR\n")
    print(err["errors"][0]["traceback"], flush=True)
    raise
PY
"""

    remote_script_path = g5k.write_remote_script(remote_workdir, "run.sh", run_sh)

    jobid, submit_out = g5k.submit_job(
        remote_dir=remote_workdir,
        script_path=remote_script_path,
        gpu_request=GPU_REQUEST,
        walltime=WALLTIME,
        host_predicate=HOST_PREDICATE,
    )

    print("\n" + submit_out.strip())
    print("jobid:", jobid)

    remote_answer_path = remote_workdir + "/answer.txt"
    remote_status_path = remote_workdir + "/status.txt"

    state, answer = g5k.wait_for_answer(
        jobid=jobid,
        remote_dir=remote_workdir,
        answer_path=remote_answer_path,
        status_path=remote_status_path,
        poll_seconds=POLL_SECONDS,
        max_wait_minutes=MAX_WAIT_MINUTES,
    )

    print("\nFinal state:", state)
    print("\nRemote summary:\n", answer)

    time.sleep(8)

    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    local_job_dir = LOCAL_JOBS_ROOT / ("job_" + jobid + "_" + ts)
    g5k.copy_remote_dir_to_local(remote_workdir, local_job_dir)
    print("\nDownloaded job folder:", local_job_dir.resolve())

    outputs_root = local_job_dir / "outputs"
    if not outputs_root.exists():
        raise RuntimeError("Missing downloaded outputs directory: " + str(outputs_root))

    for local_path in review_files:
        rel = local_path.relative_to(Path.cwd())
        rel_posix = rel.as_posix()

        src_json = outputs_root / (rel_posix + ".separated_reviews.json")
        src_txt = outputs_root / (rel_posix + ".separated_reviews.txt")
        src_err = outputs_root / (rel_posix + ".separated_reviews.error.txt")

        dst_base_dir = FINAL_OUTPUTS_ROOT / rel.parent
        dst_base_dir.mkdir(parents=True, exist_ok=True)

        dst_json = dst_base_dir / (local_path.stem + ".separated_reviews.json")
        dst_txt = dst_base_dir / (local_path.stem + ".separated_reviews.txt")
        dst_err = dst_base_dir / (local_path.stem + ".separated_reviews.error.txt")

        if src_json.exists():
            shutil.copy2(src_json, dst_json)
            print("Wrote:", dst_json.resolve())

        if src_txt.exists():
            shutil.copy2(src_txt, dst_txt)
            print("Wrote:", dst_txt.resolve())

        if src_err.exists():
            shutil.copy2(src_err, dst_err)
            print("Wrote:", dst_err.resolve())

    print("\nDone.")
    print("Job artifacts root :", LOCAL_JOBS_ROOT.resolve())
    print("Final review output:", FINAL_OUTPUTS_ROOT.resolve())


if __name__ == "__main__":
    main()

Found review files:
 - /srv/projectspace/review_outputs/jobs/job_279475_20260427_134235/input_reviews/review_outputs/paper_reviews/paper_001/submission_review_8-Reviewer_A-(Round_1).pdf
 - /srv/projectspace/review_outputs/jobs/job_279475_20260427_134235/input_reviews/review_outputs/paper_reviews/paper_001/submission_review_8-Reviewer_B-(Round_2).pdf
 - /srv/projectspace/review_outputs/jobs/job_279475_20260427_134235/input_reviews/review_outputs/paper_reviews/paper_001/submission_review_8-Reviewer_C-(Round_1).pdf
 - /srv/projectspace/review_outputs/jobs/job_279475_20260427_134235/input_reviews/review_outputs/paper_reviews/paper_002/submission_review_47-Reviewer_A-(Round_1).pdf
 - /srv/projectspace/review_outputs/jobs/job_279475_20260427_134235/input_reviews/review_outputs/paper_reviews/paper_002/submission_review_47-Reviewer_B-(Round_1).pdf
 - /srv/projectspace/review_outputs/jobs/job_279475_20260427_134235/input_reviews/review_outputs/paper_reviews/paper_003/submission_review_7-Reviewe

In [3]:
from pathlib import Path

root = Path("./JDH/review_outputs/.")
print("root exists:", root.exists(), root.resolve())

for p in sorted(root.rglob("*")):
    if p.is_file():
        print(p)

root exists: False /srv/projectspace/JDH/review_outputs


In [4]:
print('done')

done
